**Nombre: José Alexander Muñoz Delgado**

**Situación 1 — Outlier en `dias_ausencia_anio`**

Hay un trabajador con 187 días de ausencia en el año. Un año laboral en Colombia tiene aproximadamente 240 días hábiles.

Tarea: detecten ese outlier con IQR, decidan qué hacer con él y justifiquen su decisión en un comentario en el notebook.

```python
# Detecten el outlier con IQR
# Decidan: ¿eliminar la fila, corregir el valor, o conservarlo?
# Escriban la justificación como comentario en esta celda
```

**Situación 2 — Decisión de imputación para `nivel_estres`**

Volvamos a `nivel_estres`. Hay 28 nulos. Antes de imputar, comparen: ¿el nivel de estrés promedio difiere entre Juniors y Seniors?

Tarea: calculen la media de estrés por nivel, decidan si imputar con la media global o por grupo, y ejecuten la imputación.

```python
# Calculen media de nivel_estres por nivel
# Decidan la estrategia de imputación
# Ejecuten y verifiquen que no queden nulos
```

**Situación 3 — Reflexión sin código**

¿Qué columna del dataset les preocupa más en términos de calidad de datos y por qué? Escriban una respuesta de 2–3 líneas en una celda Markdown del notebook.

In [2]:
import pandas as pd
import numpy as np

#print(pd.__version__)

df = pd.read_csv("healthcheck_colombia.csv")
print(df.shape)
df.head(10)

(350, 13)


,id_trabajador,edad,ciudad,cargo,sector,nivel,anios_experiencia,horas_semana,salario_cop,nivel_estres,dias_ausencia_anio,satisfaccion_laboral,fecha_ingreso
0,1001,28,Bogotá,QA Engineer,Fintech,Junior,1,48.0,10859922.0,2.0,10.0,6.0,01/01/2018
1,1002,41,Barranquilla,UX Designer,Consultoría,Junior,9,56.0,6190583.0,8.0,6.0,8.0,04/01/2018
2,1003,50,Barranquilla,Desarrollador Junior,E-commerce,Lead,11,50.0,4099908.0,6.0,4.0,6.0,07/01/2018
3,1004,36,Bogotá,UX Designer,Healthtech,Senior,16,46.0,4664412.0,4.0,16.0,9.0,10/01/2018
4,1005,32,Medellín,Desarrollador Junior,Healthtech,Mid,7,53.0,7809792.0,5.0,19.0,9.0,13/01/2018
5,1006,29,Bogotá,Data Analyst,Edtech,Mid,10,58.0,4853847.0,8.0,8.0,2.0,16/01/2018
6,1007,50,Barranquilla,QA Engineer,Edtech,Mid,2,43.0,5392478.0,3.0,13.0,10.0,19/01/2018
7,1008,42,Bucaramanga,DevOps,Consultoría,Junior,5,45.0,NaN,NaN,11.0,1.0,22/01/2018
8,1009,28,Bucaramanga,Product Manager,Consultoría,Lead,8,49.0,8449852.0,4.0,5.0,2.0,25/01/2018
9,1010,47,Cali,Desarrollador Junior,Fintech,Mid,5,46.0,2529725.0,1.0,16.0,5.0,28/01/2018


In [8]:
# Situación 1

# Calcular Q1, Q3 e IQR
Q1 = df['dias_ausencia_anio'].quantile(0.25)
Q3 = df['dias_ausencia_anio'].quantile(0.75)
IQR = Q3 - Q1

# Límites
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Límite superior: {limite_superior}")

# Outliers
outliers = df[
    (df['dias_ausencia_anio'] < limite_inferior) |
    (df['dias_ausencia_anio'] > limite_superior)
]
print(outliers[['dias_ausencia_anio']])

# Se identificó un valor atípico en la fila 289 con 187 días de ausencia anual. 
# Mediante el método IQR se obtuvo un límite superior de 27.5, por lo que el valor 187 
# supera ampliamente dicho umbral y se clasifica como outlier. Sin embargo, se decide conservarlo, 
# ya que dentro del contexto laboral puede corresponder a una incapacidad médica 
# prolongada o una licencia extensa, por lo que no necesariamente representa un error en los datos.

Q1: 5.0
Q3: 14.0
IQR: 9.0
Límite superior: 27.5
     dias_ausencia_anio
289               187.0


In [5]:
# Situación 2

# Media de estrés por nivel
media_por_nivel = df.groupby('nivel')['nivel_estres'].mean()
print(media_por_nivel)

nivel
Junior    5.099099
Lead      5.485714
Mid       5.273585
Senior    5.385714
Name: nivel_estres, dtype: float64


In [ ]:
# Se decide imputar por grupo según el nivel laboral,
# ya que las medias presentan ligeras diferencias
# entre Junior, Mid, Senior y Lead.

df['nivel_estres'] = df.groupby('nivel')['nivel_estres']\
    .transform(lambda x: x.fillna(x.mean()))

# Se valida el DataFrame que no tenga datos nulos
print(df['nivel_estres'].isnull().sum())

# Aunque en el ejercicio se menciona comparar Junior y Senior, 
# al revisar la columna nivel también se encontraron las categorías Mid y Lead. 
# Por eso se decidió tener en cuenta todos los niveles que aparecen en los datos y 
# hacer la imputación por grupo, ya que el promedio del nivel de estrés cambia 
# un poco entre cada categoría.

0


Situación 3

La columna que más preocupa es dias_ausencia_anio, ya que presenta un valor atípico bastante alto que puede afectar medidas como la media y la desviación estándar. También nivel_estres requiere atención por los valores nulos encontrados, porque si no se imputan correctamente podrían alterar el análisis posterior.